In [2]:
list.of.packages = c("tidyverse", "bigrquery")
install.packages(list.of.packages)

library(tidyverse)
library(bigrquery)

# The below queries represent dataset AllOfUsControlledTierDatasetv9 in cohort hhtepi at 20260817_094518.
# This file saves data into ~/workspace/hht-epi-bucket/bq_exports

# If EXPORT_BUCKET is not set in your environment, uncomment and update the line below
# with your actual Cloud Storage bucket name from the Resources tab in your workspace.
# Sys.setenv(EXPORT_BUCKET = "gs://your-actual-bucket-name")



Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [3]:
# Domain person
person_sql = r"(

    SELECT
        st.birth_datetime,
        st.ethnicity_concept_id,
        dt0.concept_name AS T_DISP_ethnicity,
        st.gender_concept_id,
        dt1.concept_name AS T_DISP_gender,
        st.person_id,
        st.race_concept_id,
        dt2.concept_name AS T_DISP_race,
        st.self_reported_category_concept_id,
        dt3.concept_name AS T_DISP_self_reported_category,
        st.sex_at_birth_concept_id,
        dt4.concept_name AS T_DISP_sex_at_birth 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.person AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.ethnicity_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.gender_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.race_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.self_reported_category_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_person 
        WHERE
            (id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "person",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person",
  "person_*.csv")
message(str_glue('The data will be written to {person_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", person_sql, billing = "wb-lucent-greens-2488"),
  person_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

person_df <- read_bq_export_from_workspace_bucket(person_path)
dim(person_df)
head(person_df, 5)


The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/person/20260820/person/person_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/person/20260820/person/person_000000000000.csv.



[1] 1264   12

birth_datetime,ethnicity_concept_id,T_DISP_ethnicity,gender_concept_id,T_DISP_gender,person_id,race_concept_id,T_DISP_race,self_reported_category_concept_id,T_DISP_self_reported_category,sex_at_birth_concept_id,T_DISP_sex_at_birth
<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1941-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,45878463,Female,2175828,2000000008,More than one population,2000000008,More than one population,0,No matching concept
1962-06-15 00:00:00 UTC,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,2565348,45882607,None of these,45882607,None of these,45880669,Male
1975-06-15 00:00:00 UTC,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,6025395,45882607,None of these,45882607,None of these,45880669,Male
1955-06-15 00:00:00 UTC,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,1508067,45882607,None of these,45882607,None of these,45880669,Male
1989-06-15 00:00:00 UTC,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,9106930,45882607,None of these,45882607,None of these,45880669,Male


In [4]:
# Domain ingredientOccurrence
ingredientOccurrence_sql = r"(

    SELECT
        st.days_supply,
        st.dose_unit_source_value,
        st.drug_concept_id,
        st.drug_exposure_end_datetime,
        st.drug_exposure_start_datetime,
        st.drug_source_concept_id,
        st.drug_source_value,
        st.drug_type_concept_id,
        dt0.concept_name AS T_DISP_drug_type_concept_name,
        st.lot_number,
        st.person_id,
        st.quantity,
        st.refills,
        st.route_concept_id,
        dt1.concept_name AS T_DISP_route_concept_name,
        st.route_source_value,
        st.sig,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.stop_reason,
        st.verbatim_end_date,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.drug_exposure AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.drug_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.route_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.drug_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.drug_concept_id 
    WHERE
        st.drug_exposure_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_ingredientOccurrence_person_id 
        WHERE
            ((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))))) 
                AND (drug_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_ingredientConcept_default 
                WHERE
                    ancestor IN (1303425, 1397141, 21601078, 43014237)  -- + pomalidomide 
                UNION
                ALL SELECT
                    1303425 
                UNION
                ALL SELECT
                    1397141 
                UNION
                ALL SELECT
                    21601078
UNION
ALL SELECT
43014237))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
ingredientOccurrence_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "ingredientOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "ingredientOccurrence",
  "ingredientOccurrence_*.csv")
message(str_glue('The data will be written to {ingredientOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", ingredientOccurrence_sql, billing = "wb-lucent-greens-2488"),
  ingredientOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {ingredientOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

ingredientOccurrence_df <- read_bq_export_from_workspace_bucket(ingredientOccurrence_path)
dim(ingredientOccurrence_df)
head(ingredientOccurrence_df, 5)


The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/ingredientOccurrence/20260820/ingredientOccurrence/ingredientOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/ingredientOccurrence/20260820/ingredientOccurrence/ingredientOccurrence_000000000000.csv.

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”


[1] 6122   26

days_supply,dose_unit_source_value,drug_concept_id,drug_exposure_end_datetime,drug_exposure_start_datetime,drug_source_concept_id,drug_source_value,drug_type_concept_id,T_DISP_drug_type_concept_name,lot_number,⋯,sig,T_DISP_source_concept_code,T_DISP_source_concept_name,T_DISP_source_vocabulary,T_DISP_standard_concept_code,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,stop_reason,verbatim_end_date,visit_occurrence_id
<dbl>,<lgl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<lgl>,⋯,<lgl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<lgl>,<date>,<dbl>
NA,NA,43014251,2023-03-02 00:00:00 UTC,2023-02-01 00:00:00 UTC,NA,NA,32838,EHR prescription,NA,⋯,NA,NA,NA,NA,1369734,pomalidomide 4 MG Oral Capsule,RxNorm,NA,NA,NA
NA,NA,43014251,2023-03-01 00:00:00 UTC,2023-01-31 00:00:00 UTC,NA,NA,32838,EHR prescription,NA,⋯,NA,NA,NA,NA,1369734,pomalidomide 4 MG Oral Capsule,RxNorm,NA,NA,NA
NA,NA,43014251,2023-03-02 00:00:00 UTC,2023-02-01 00:00:00 UTC,NA,NA,32838,EHR prescription,NA,⋯,NA,NA,NA,NA,1369734,pomalidomide 4 MG Oral Capsule,RxNorm,NA,NA,NA
NA,NA,43014251,2023-04-06 00:00:00 UTC,2023-03-08 00:00:00 UTC,NA,NA,32838,EHR prescription,NA,⋯,NA,NA,NA,NA,1369734,pomalidomide 4 MG Oral Capsule,RxNorm,NA,NA,NA
NA,NA,43014251,2021-10-26 00:00:00 UTC,2021-07-08 09:04:00 UTC,NA,NA,38000177,Prescription written,NA,⋯,NA,NA,NA,NA,1369734,pomalidomide 4 MG Oral Capsule,RxNorm,NA,NA,3.8e+16


In [5]:
# Domain procedureOccurrence
procedureOccurrence_sql = r"(

    SELECT
        st.modifier_concept_id,
        dt0.concept_name AS T_DISP_modifier_concept_name,
        st.modifier_source_value,
        st.person_id,
        st.procedure_concept_id,
        st.procedure_datetime,
        st.procedure_source_concept_id,
        st.procedure_source_value,
        st.procedure_type_concept_id,
        dt1.concept_name AS T_DISP_procedure_type_concept_name,
        st.quantity,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.procedure_occurrence AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.modifier_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.procedure_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.procedure_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.procedure_concept_id 
    WHERE
        st.procedure_occurrence_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_procedureOccurrence_person_id 
        WHERE
            ((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))))) 
                AND (procedure_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_procedureConcept_default 
                WHERE
                    ancestor IN (4022173, 4046732, 4122427, 4125928, 4144919, 4218549, 4220838, 4249893, 4262407, 4294382, 37156556, 40485932, 4049250, 4138020)  -- + septodermoplasty, laser therapy of internal nose 
                UNION
                ALL SELECT
                    4022173 
                UNION
                ALL SELECT
                    4046732 
                UNION
                ALL SELECT
                    4122427 
                UNION
                ALL SELECT
                    4125928 
                UNION
                ALL SELECT
                    4144919 
                UNION
                ALL SELECT
                    4218549 
                UNION
                ALL SELECT
                    4220838 
                UNION
                ALL SELECT
                    4249893 
                UNION
                ALL SELECT
                    4262407 
                UNION
                ALL SELECT
                    4294382 
                UNION
                ALL SELECT
                    37156556 
                UNION
                ALL SELECT
                    40485932
UNION
ALL SELECT
4049250
UNION
ALL SELECT
4138020))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedureOccurrence_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "procedureOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedureOccurrence",
  "procedureOccurrence_*.csv")
message(str_glue('The data will be written to {procedureOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", procedureOccurrence_sql, billing = "wb-lucent-greens-2488"),
  procedureOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedureOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

procedureOccurrence_df <- read_bq_export_from_workspace_bucket(procedureOccurrence_path)
dim(procedureOccurrence_df)
head(procedureOccurrence_df, 5)


The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/procedureOccurrence/20260820/procedureOccurrence/procedureOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/procedureOccurrence/20260820/procedureOccurrence/procedureOccurrence_000000000000.csv.



[1] 533  18

modifier_concept_id,T_DISP_modifier_concept_name,modifier_source_value,person_id,procedure_concept_id,procedure_datetime,procedure_source_concept_id,procedure_source_value,procedure_type_concept_id,T_DISP_procedure_type_concept_name,quantity,T_DISP_source_concept_code,T_DISP_source_concept_name,T_DISP_source_vocabulary,T_DISP_standard_concept_code,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,visit_occurrence_id
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
NA,NA,NA,4535982,40485932,2022-08-15 06:00:00 UTC,40485932,445982009,32817,EHR,1,445982009,Endoscopy and biopsy of upper gastrointestinal tract,SNOMED,445982009,Endoscopy and biopsy of upper gastrointestinal tract,SNOMED,5.100000e+16
0,No matching concept,No matching concept,7784701,40485932,2012-03-29 11:25:00 UTC,2002688,45.16,38000252,Inpatient header - 2nd position,1,45.16,Esophagogastroduodenoscopy [EGD] with closed biopsy,ICD9Proc,445982009,Endoscopy and biopsy of upper gastrointestinal tract,SNOMED,6.700003e+16
0,No matching concept,No matching concept,1051898,40485932,2013-10-23 05:00:00 UTC,2002688,45.16,44786631,Secondary Procedure,NA,45.16,Esophagogastroduodenoscopy [EGD] with closed biopsy,ICD9Proc,445982009,Endoscopy and biopsy of upper gastrointestinal tract,SNOMED,4.009531e+16
0,No matching concept,No matching concept,1475441,40485932,2014-12-31 00:00:00 UTC,2002688,45.16,0,No matching concept,NA,45.16,Esophagogastroduodenoscopy [EGD] with closed biopsy,ICD9Proc,445982009,Endoscopy and biopsy of upper gastrointestinal tract,SNOMED,3.800000e+16
NA,NA,NA,9752437,40485932,1993-01-15 06:00:00 UTC,2002688,45.16,32810,Claim,NA,45.16,Esophagogastroduodenoscopy [EGD] with closed biopsy,ICD9Proc,445982009,Endoscopy and biopsy of upper gastrointestinal tract,SNOMED,7.100000e+16


In [6]:
# Domain zipcodeSocioeconomic
zipcodeSocioeconomic_sql = r"(

    SELECT
        american_community_survey_year,
        assisted_income,
        deprivation_index,
        high_school_education,
        id,
        median_income,
        no_health_insurance,
        observation_datetime_str,
        observation_datetime_utc,
        person_id,
        poverty,
        vacant_housing,
        zip_code 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_zipcodeSocioeconomic 
    WHERE
        (
            person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))
            ) 
            OR (
                (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))
                ) 
                OR (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))
                )
            ))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zipcodeSocioeconomic_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "zipcodeSocioeconomic",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zipcodeSocioeconomic",
  "zipcodeSocioeconomic_*.csv")
message(str_glue('The data will be written to {zipcodeSocioeconomic_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", zipcodeSocioeconomic_sql, billing = "wb-lucent-greens-2488"),
  zipcodeSocioeconomic_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zipcodeSocioeconomic_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

zipcodeSocioeconomic_df <- read_bq_export_from_workspace_bucket(zipcodeSocioeconomic_path)
dim(zipcodeSocioeconomic_df)
head(zipcodeSocioeconomic_df, 5)


The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/zipcodeSocioeconomic/20260820/zipcodeSocioeconomic/zipcodeSocioeconomic_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/zipcodeSocioeconomic/20260820/zipcodeSocioeconomic/zipcodeSocioeconomic_000000000000.csv.



[1] 1208   13

american_community_survey_year,assisted_income,deprivation_index,high_school_education,id,median_income,no_health_insurance,observation_datetime_str,observation_datetime_utc,person_id,poverty,vacant_housing,zip_code
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
2017,5.165908,0.1625441,96.18835,52794,126454.5,2.032504,2019-03-19 16:21:22,2019-03-19 16:21:22 UTC,2495533,6.550429,5.271112,024**
2017,5.165908,0.1625441,96.18835,431517,126454.5,2.032504,2022-09-09 14:36:31,2022-09-09 14:36:31 UTC,9194415,6.550429,5.271112,024**
2017,5.165908,0.1625441,96.18835,608707,126454.5,2.032504,2024-01-24 22:30:01,2024-01-24 22:30:01 UTC,8129242,6.550429,5.271112,024**
2017,5.165908,0.1625441,96.18835,704156,126454.5,2.032504,2019-07-17 18:06:57,2019-07-17 18:06:57 UTC,3446758,6.550429,5.271112,024**
2017,5.165908,0.1625441,96.18835,685135,126454.5,2.032504,2020-08-27 19:09:22,2020-08-27 19:09:22 UTC,2325217,6.550429,5.271112,024**


In [7]:
# Domain conditionOccurrence
conditionOccurrence_sql = r"(

    SELECT
        st.condition_concept_id,
        st.condition_end_datetime,
        st.condition_source_concept_id,
        st.condition_source_value,
        st.condition_start_datetime,
        st.condition_status_concept_id,
        dt0.concept_name AS T_DISP_condition_status_concept_name,
        st.condition_status_source_value,
        st.condition_type_concept_id,
        dt1.concept_name AS T_DISP_condition_type_concept_name,
        st.person_id,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.stop_reason,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.condition_occurrence AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.condition_status_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.condition_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.condition_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.condition_concept_id 
    WHERE
        st.condition_occurrence_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_conditionOccurrence_person_id 
        WHERE
            (((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))))) 
                AND (condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor IN (192671, 432875, 436659, 439777, 439847, 4049224, 4051478, 4096682, 4119302, 4121803, 4146209, 37162759, 37170305, 4224145)  -- + arteriovenous malformation of liver 
                UNION
                ALL SELECT
                    192671 
                UNION
                ALL SELECT
                    432875 
                UNION
                ALL SELECT
                    436659 
                UNION
                ALL SELECT
                    439777 
                UNION
                ALL SELECT
                    439847 
                UNION
                ALL SELECT
                    4049224 
                UNION
                ALL SELECT
                    4051478 
                UNION
                ALL SELECT
                    4096682 
                UNION
                ALL SELECT
                    4119302 
                UNION
                ALL SELECT
                    4121803 
                UNION
                ALL SELECT
                    4146209 
                UNION
                ALL SELECT
                    37162759 
                UNION
                ALL SELECT
                    37170305
UNION
ALL SELECT
4224145))) 
                OR (((person_id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                WHERE
                    condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504))) 
                    OR ((person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))))) 
                    AND (condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504)))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
conditionOccurrence_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "conditionOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "conditionOccurrence",
  "conditionOccurrence_*.csv")
message(str_glue('The data will be written to {conditionOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", conditionOccurrence_sql, billing = "wb-lucent-greens-2488"),
  conditionOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {conditionOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

conditionOccurrence_df <- read_bq_export_from_workspace_bucket(conditionOccurrence_path)
dim(conditionOccurrence_df)
head(conditionOccurrence_df, 5)


The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/conditionOccurrence/20260820/conditionOccurrence/conditionOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports/conditionOccurrence/20260820/conditionOccurrence/conditionOccurrence_000000000000.csv.



[1] 26092    19

condition_concept_id,condition_end_datetime,condition_source_concept_id,condition_source_value,condition_start_datetime,condition_status_concept_id,T_DISP_condition_status_concept_name,condition_status_source_value,condition_type_concept_id,T_DISP_condition_type_concept_name,person_id,T_DISP_source_concept_code,T_DISP_source_concept_name,T_DISP_source_vocabulary,T_DISP_standard_concept_code,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,stop_reason,visit_occurrence_id
<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<lgl>,<dbl>
4232181,NA,35208214,K26.4,2023-11-20 19:57:54 UTC,0,No matching concept,No matching concept,0,No matching concept,2574911,K26.4,Chronic or unspecified duodenal ulcer with hemorrhage,ICD10CM,89469000,Chronic duodenal ulcer with hemorrhage,SNOMED,NA,NA
4232181,NA,35208214,K26.4,2024-07-26 17:03:38 UTC,0,No matching concept,No matching concept,0,No matching concept,2574911,K26.4,Chronic or unspecified duodenal ulcer with hemorrhage,ICD10CM,89469000,Chronic duodenal ulcer with hemorrhage,SNOMED,NA,NA
4232181,NA,35208214,K26.4,2024-01-25 02:39:00 UTC,0,No matching concept,No matching concept,0,No matching concept,2574911,K26.4,Chronic or unspecified duodenal ulcer with hemorrhage,ICD10CM,89469000,Chronic duodenal ulcer with hemorrhage,SNOMED,NA,NA
4232181,NA,35208214,K26.4,2020-07-16 10:26:37 UTC,0,No matching concept,No matching concept,0,No matching concept,2574911,K26.4,Chronic or unspecified duodenal ulcer with hemorrhage,ICD10CM,89469000,Chronic duodenal ulcer with hemorrhage,SNOMED,NA,NA
4232181,NA,35208214,K26.4,2021-05-27 01:08:37 UTC,0,No matching concept,No matching concept,0,No matching concept,2574911,K26.4,Chronic or unspecified duodenal ulcer with hemorrhage,ICD10CM,89469000,Chronic duodenal ulcer with hemorrhage,SNOMED,NA,NA
